# SDE-Net diretto t+6h — evento sahariano di aprile 2019

Analisi post-hoc del modello già addestrato. Il notebook **non rilancia il training** e usa tutte le predizioni disponibili, senza campionamento.

Evento esterno di riferimento: grande intrusione di polvere sahariana iniziata il 21 aprile 2019, osservata da NASA/MODIS sull'Europa il 23 aprile e fino alle Alpi il 25 aprile: https://modis.gsfc.nasa.gov/gallery/individual.php?db_date=2019-04-26

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.posthoc_outputs as posthoc_outputs
import physiq_pv.experiments.sde_pipeline as pipe

importlib.reload(posthoc_outputs)
pipe = importlib.reload(pipe)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
FORECAST_HORIZON = 6
out_dir = ROOT / 'outputs' / RUN_NAME
predictions_path = out_dir / 'predictions.csv'
posthoc_dir = out_dir / 'posthoc_by_horizon' / f't_plus_{FORECAST_HORIZON}'
reference_peak_path = posthoc_dir / 'reference_production_peaks.csv'
pvgis_2019_path = Path('/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance/piedmont_pvgis_2019.nc')

for required in (
    predictions_path,
    reference_peak_path,
    pvgis_2019_path,
):
    if not required.is_file():
        raise FileNotFoundError(required)

print('Run SDE-Net:', out_dir)
print('Predizioni:', predictions_path)
print('PVGIS 2019:', pvgis_2019_path)

## 1. Evidenza fisica nei PVGIS

La POA viene ricostruita nello stesso modo della pipeline: `direct_irradiance_tilted + diffuse_irradiance_tilted`. Il campo `solar_irradiance_poa` originale non viene utilizzato.

In [ ]:
variables = [
    'direct_irradiance_tilted',
    'diffuse_irradiance_tilted',
    'temperature_2m',
    'wind_speed_10m',
    'pv_power_output',
]
with xr.open_dataset(pvgis_2019_path) as dataset:
    april = dataset[variables].sel(
        time=slice('2019-04-18', '2019-04-30T23:59:59')
    ).load()

timestamps = pd.to_datetime(april.time.values)
dates = pd.Index(timestamps.date)
physical_rows = []
for date in dates.unique():
    mask = dates == date
    direct = april.direct_irradiance_tilted.values[:, mask]
    diffuse = april.diffuse_irradiance_tilted.values[:, mask]
    poa = direct + diffuse
    daylight = poa > 10.0
    node_energy = poa.sum(axis=1) / 1000.0
    physical_rows.append({
        'date': pd.Timestamp(date),
        'poa_kwh_m2': float(node_energy.mean()),
        'direct_kwh_m2': float(direct.sum(axis=1).mean() / 1000.0),
        'diffuse_kwh_m2': float(diffuse.sum(axis=1).mean() / 1000.0),
        'diffuse_fraction': float((diffuse[daylight] / poa[daylight]).mean()),
        'poa_spatial_std_kwh_m2': float(node_energy.std()),
        'temperature_daylight_mean': float(
            april.temperature_2m.values[:, mask][daylight].mean()
        ),
        'pv_kwh_mean': float(
            april.pv_power_output.values[:, mask].sum(axis=1).mean() / 1000.0
        ),
    })

physical = pd.DataFrame(physical_rows).set_index('date')
display(physical)

physical_figure_dir = out_dir / 'figures' / 'april_dust_event'
physical_figure_dir.mkdir(parents=True, exist_ok=True)
physical_figure_path = physical_figure_dir / 'april_irradiance_components.png'

fig, axis = plt.subplots(figsize=(12, 5))
axis.bar(
    physical.index,
    physical['direct_kwh_m2'],
    label='direct tilted',
    color='tab:orange',
)
axis.bar(
    physical.index,
    physical['diffuse_kwh_m2'],
    bottom=physical['direct_kwh_m2'],
    label='diffuse tilted',
    color='tab:blue',
)
axis.axvspan(pd.Timestamp('2019-04-23'), pd.Timestamp('2019-04-26'), color='red', alpha=0.10)
axis.set(
    title='PVGIS Piemonte — componenti giornaliere della POA',
    ylabel='Energia media regionale [kWh/m²]',
    xlabel='Data',
)
axis.legend()
axis.grid(axis='y', alpha=0.25)
fig.autofmt_xdate()
fig.savefig(physical_figure_path, dpi=140, bbox_inches='tight')
plt.show()
print('Grafico:', physical_figure_path)

## 2. Risposta temporale di SDE-Net

La finestra 21–27 aprile include l'ingresso nell'evento, i giorni più severi e il ritorno alla normalità. La diagnostica usa tutte le 1.149 località e mostra previsione, errore, copertura, frazione regionale MTGFlow e componenti di incertezza.

In [ ]:
april_event = pipe.build_extreme_event_diagnostic(
    str(out_dir),
    start='2019-04-21',
    end='2019-04-28',
    figure_subdir='events/t_plus_6/april_dust_event',
    horizon_hours=FORECAST_HORIZON,
)

print(f"Righe usate (nessun campionamento): {april_event['rows_used']:,}")
print('Soglia regionale MAM:', april_event['regional_threshold'])
display(april_event['summary'])
display(april_event['hourly'])
display(Image(filename=str(april_event['figure_path'])))
print('CSV orario:', april_event['hourly_path'])
print('CSV riepilogo:', april_event['summary_path'])

## 3. Normale 2019 vs 23, 24, 25 e 26 aprile

Il confronto usa esclusivamente righe diurne valide, tutte le località e nessun campionamento. Le categorie sono separate nelle fasce 0–20%, 20–40%, 40–60%, 60–80% e 80–100% della potenza di riferimento. Il 26 aprile è mantenuto separato per osservare il possibile trascinamento della finestra MTGFlow.

In [ ]:
april_comparison = pipe.build_extreme_event_comparison_figures(
    str(out_dir),
    event_dates=(
        '2019-04-23',
        '2019-04-24',
        '2019-04-25',
        '2019-04-26',
    ),
    comparison_name='april_dust_23_26_t_plus_6',
    figure_subdir='events/t_plus_6/april_dust_event',
    horizon_hours=FORECAST_HORIZON,
    reference_peak_path=reference_peak_path,
)

metrics = april_comparison['metrics']
display(metrics)
for metric in ('mae', 'rmse', 'picp', 'nmpil', 'clc'):
    print(f'\n=== {metric.upper()} ===')
    display(metrics.pivot(index='bin', columns='category', values=metric))

print('CSV metriche:', april_comparison['metrics_path'])
print('Grafici creati:', len(april_comparison['figure_paths']))
for figure_path in april_comparison['figure_paths'].values():
    display(Image(filename=str(figure_path)))

## 4. Lettura dei risultati

Controllare in particolare:

- se MAE e RMSE aumentano rispetto ai timestamp normali nelle stesse fasce di produzione;
- se il PICP resta vicino al 95% oppure crolla durante l'ingresso nell'evento;
- se l'aumento della banda è aleatorico o epistemico;
- se il 26 aprile resta raro ma torna facile da prevedere, indicando trascinamento della finestra MTGFlow più che difficoltà fisica persistente;
- se gli errori maggiori coincidono con il passaggio da irradiazione diretta a quasi interamente diffusa.